# One function for stored and live data

screamer provides functions that transform a sequence of values into a new sequence. A rolling mean, for instance, replaces each value with the average of the recent ones, turning a noisy series into a smoother one.

A function is defined once and then applied to data in whatever form you have. Give it a NumPy array and it returns the transformed array. Give it values one at a time as they arrive from a live feed and it returns each result as it lands. The output does not depend on how the data is delivered, so a function you develop and test on stored history runs unchanged on a live stream.

Every example below uses one function, a five-sample rolling mean. The [polymorphic API reference](../polymorphic_api.md) gives the precise rule for each input type.

In [ ]:
import numpy as np
from screamer import RollingMean

data = [3.2, 1.8, 3.4, 2.9, 3.6, 3.1, 4.0, 3.3, 2.7, 3.5]   # ten readings
mean5 = RollingMean(5)   # a five-sample rolling mean

## An array

Pass a one-dimensional array and get back an array of the same length. The first four entries are `NaN`, because the window needs five samples before it can report a mean.

In [ ]:
mean5(np.array(data)).round(3)

## Several series at once

The first axis is time. Any further axes are separate series carried alongside each other. A `(10, 3)` array is three rolling means worked out together, and you write no loop of your own.

In [ ]:
series = np.random.default_rng(0).standard_normal((10, 3))
mean5(series).shape

## A Python list

Give it a list and a list comes back, which is convenient when your data is not already in NumPy.

In [ ]:
out = mean5(data)
print(type(out).__name__)          # list
print([round(v, 3) for v in out])

## One value at a time

When values arrive one by one, call the operator on each as it shows up. Every call returns the rolling mean up to that point, and the operator remembers what it has seen so the next call continues the same window. A fresh operator starts a fresh stream.

In [ ]:
live = RollingMean(5)
[round(live(x), 3) for x in data]   # one value in, one value out

## A lazy iterator

Hand the operator a generator and it hands back a lazy iterator. Nothing is computed until you pull a value, and each value you pull draws exactly one input through. This is the form to reach for inside an event loop, or for a stream with no fixed length.

In [ ]:
stream = RollingMean(5)(x for x in data)
print(type(stream).__name__)                 # a lazy iterator; nothing has run yet
[round(next(stream), 3) for _ in range(6)]   # pull six values on demand

## Building a graph

One more input type changes what a call does. Pass a graph input in place of data and the operator records a step in a computation graph rather than running straight away. This is how operators compose into a DAG. The [computational DAG notebook](09-computational-dag.ipynb) covers it.